In [ ]:
import pandas as pd
from datetime import datetime
import glob

def get_path():
  list_over_1000 = glob.glob('input/reports.disciplinary.disciplinarySanctions*.xlsx')
  list_under_1000 = glob.glob('input/DisciplinarySanctions_*.xlsx')
  if len(list_over_1000) > 0:
    return list_over_1000[0]
  else:
    return list_under_1000[0]

sanctions_path = get_path()

today = datetime.today().strftime('%Y-%m-%d')

pending_sanctions = pd.read_excel(sanctions_path)

pending_sanctions = pending_sanctions[pending_sanctions['Competition type'].str.contains('OPL Cup|Ontario')]

pending_sanctions = pending_sanctions[pending_sanctions['Case status'].str.contains('ENTERED')]

pending_sanctions['Pending since'] = pending_sanctions.apply(lambda x: pd.to_datetime(x['In status since (EDT)']).date(), axis=1)

pending_sanctions = pending_sanctions[['Parent Club', 'Person name', 'Offender type', 'Pending since']].sort_values(by=['Pending since'])

pending_sanctions = pending_sanctions.rename(columns={'Parent Club': 'Club', 'Person name': 'Name', 'Offender type': 'Person type'})

last_updated = pd.DataFrame({ 'Club': ['Last updated:'], 'Name': [today]})

pending_sanctions = pd.concat([pending_sanctions, last_updated], ignore_index=True)

pending_sanctions.to_csv('output/opl_pending_suspensions.csv', index=False)

pending_sanctions
